In [ ]:
import Pkg
Pkg.update()
Pkg.add("ProfileView")
Pkg.add("ProfileCanvas")

In [ ]:
import Pkg
using MLDatasets
using Flux: onehotbatch, onecold
using Statistics
using LinearAlgebra
using Random
using ProfileCanvas

train_data = MLDatasets.FashionMNIST(split=:train)
test_data  = MLDatasets.FashionMNIST(split=:test)

X_train = Float32.(reshape(train_data.features, 28, 28, 1, :))
X_test  = Float32.(reshape(test_data.features, 28, 28, 1, :))

Y_train_raw = train_data.targets
Y_test_raw  = test_data.targets

Y_train = Float32.(onehotbatch(Y_train_raw, 0:9))
Y_test  = Float32.(onehotbatch(Y_test_raw, 0:9))

println("Wymiary: ", size(X_train))

include("clothesolver.jl") 

# 1. Definicja
my_net_def = Chain(
  Conv((3, 3), 1 => 6, bias=false),
  MaxPool((2, 2)),
  Conv((3, 3), 6 => 16, bias=false),
  MaxPool((2, 2)),
  Flatten(),               
  Dense(400 => 84, relu), # Pamiętaj o 400 z Shape Inference!
  Dropout(0.4),
  Dense(84 => 10)
)

# 2. Alokacja Memory Pool
my_model = build_model(my_net_def, (28, 28, 1))

input_node  = alloc_act!(my_model.pool, 28, 28, 1)
target_node = alloc_act!(my_model.pool, 10)
loss_fn     = LogitCrossEntropy(my_model.pool, 10)

Wymiary: (28, 28, 1, 60000)


In [ ]:
x_sample = X_train[:, :, :, 1]
y_sample = Y_train[:, 1]

# 1. ROZGRZEWKA (Warm-up)
# Musimy odpalić to raz w ciszy, żeby Julia skompilowała kod (LLVM JIT). 
# Jeśli będziesz profilował pierwszą iterację, 99% czasu zajmie kompilator.
copyto!(input_node.data, x_sample)
copyto!(target_node.data, y_sample)

preds = forward!(my_model.chain, input_node, true)
primal!(loss_fn, preds, target_node)
loss_fn.out.grad[1] = 1.0f0
adjoint!(loss_fn, preds, target_node)
backward!(my_model.chain, input_node, true)

println("Kompilacja zakończona, start profilowania...")

# 2. PROFILOWANIE
@time ProfileCanvas.@profview for _ in 1:1000
    # W nowej architekturze musimy czyścić bufory ręcznie w pętli
    zero_a_grad!(my_model.pool)
    zero_w_grad!(my_model.pool)
    
    # Wstrzyknięcie danych do zaalokowanych węzłów I/O
    copyto!(input_node.data, x_sample)
    copyto!(target_node.data, y_sample)
    
    # Obliczenia
    preds = forward!(my_model.chain, input_node, true)
    primal!(loss_fn, preds, target_node)
    
    loss_fn.out.grad[1] = 1.0f0
    adjoint!(loss_fn, preds, target_node)
    backward!(my_model.chain, input_node, true)
end

  1.900311 seconds (659.14 k allocations: 36.071 MiB, 37.43% compilation time)
